# 🔐 CodeReviewEnv v3 — GRPO Training Notebook

**Theme:** 3.1 World Modeling → Professional Tasks

This notebook trains **Qwen3-1.7B** (thinking-mode) using GRPO + the metacognitive reward to investigate CVE vulnerabilities.
The agent learns to strategically read code, search patterns, and allocate its
"thinking budget" — reasoning deeply on suspicious files and briefly on safe ones.

### Requirements
- GPU Runtime: **T4 (16GB)** minimum, **A10G/A100** recommended
- Training time: ~3-5 hours on A10G, ~5-7 hours on T4

In [ ]:
# Cell 1: Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.17" peft accelerate bitsandbytes
!pip install -q openenv-core fastmcp datasets matplotlib

In [ ]:
# Cell 2: Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU! Switch to GPU runtime."
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"✅ GPU: {gpu} | VRAM: {vram:.1f} GB")

In [ ]:
# Cell 3: Clone the environment
!git clone https://huggingface.co/spaces/YOUR_USERNAME/code-review-env ./code-review-env
import sys
sys.path.insert(0, './code-review-env')

In [ ]:
# Cell 4: Verify environment works
from code_review_env.server.environment import CodeReviewEnvironment
from openenv.core.env_server import CallToolAction
import re

env = CodeReviewEnvironment()
obs = env.reset(difficulty='easy')
ctx = obs.metadata['context']
files = re.findall(r'• (.+?)\s+\[', ctx)

print(f'✅ Environment loaded: {len(files)} files')

# Quick investigation test
obs = env.step(CallToolAction(tool_name='read_file', arguments={'file_path': files[0]}))
print(f'✅ read_file works')

obs = env.step(CallToolAction(tool_name='flag_vulnerable', arguments={
    'file_path': files[0],
    'reasoning': 'Buffer overflow in memcpy without bounds check'
}))
print(f'✅ flag_vulnerable works')

obs = env.step(CallToolAction(tool_name='submit_report', arguments={
    'summary': 'Test investigation report', 'confidence': 'medium'
}))
result = str(obs.result.data if hasattr(obs.result, 'data') else obs.result)
score = re.search(r'TOTAL SCORE: ([\d.]+)', result)
print(f'✅ submit_report works — score: {score.group(1) if score else "?"}')
print(f'\n🎯 Environment verified! Ready to train.')

In [ ]:
# Cell 5: Run GRPO training
# This will take 3-5 hours on A10G, 5-7 hours on T4
%run train_grpo.py

In [ ]:
# Cell 6: View results
from IPython.display import Image, display
import json

display(Image('grpo_output/training_curves.png'))

with open('grpo_output/training_stats.json') as f:
    stats = json.load(f)
print(f"\n📊 Model: {stats['model']}")
print(f"   Early mean reward: {stats['early_mean']:.3f}")
print(f"   Late mean reward:  {stats['late_mean']:.3f}")
print(f"   Improvement:       {stats['improvement']:+.3f}")
print(f"   Max reward:        {stats['max_reward']:.3f}")